# EfficientNetB0 Transfer Learning for 40-Class Leaf Disease Classification

This notebook implements a comprehensive EfficientNetB0 transfer learning pipeline with two-phase training strategy (feature extraction + fine-tuning). EfficientNetB0 is a parameter-efficient architecture that scales depth, width, and resolution uniformly, achieving better accuracy with fewer parameters compared to conventional models.

## 1. Import Required Libraries

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report, precision_recall_fscore_support
import json
import os
import sys
from pathlib import Path

# Add utils to path
sys.path.insert(0, '/root/cnn_model')
from utils.model_utils import (
    build_efficientnet_b0_model,
    unfreeze_model_layers,
    evaluate_model,
    compute_class_weights_from_labels
)

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

# Display TensorFlow version
print(f"TensorFlow version: {tf.__version__}")
print(f"GPU Available: {tf.config.list_physical_devices('GPU')}")

## 2. Configuration and Data Loading

In [ ]:
# Configuration dictionary for EfficientNetB0 training
CONFIG = {
    'num_classes': 40,
    'input_shape': (224, 224, 3),
    'batch_size': 16,
    'phase1_epochs': 20,
    'phase2_epochs': 15,
    'phase1_learning_rate': 1e-4,
    'phase2_learning_rate': 1e-5,
    'optimizer': 'adam',
    'dense_units': 256,
    'dropout_rate': 0.35,
    'num_unfreeze_layers': 50,
    'data_dir': '/root/cnn_model/results/preprocessed_data',
    'models_dir': '/root/cnn_model/models/efficientnetb0',
    'logs_dir': '/root/cnn_model/results/logs',
    'plots_dir': '/root/cnn_model/results/plots',
    'confusion_matrix_dir': '/root/cnn_model/results/confusion_matrix'
}

# Create directories if they don't exist
for dir_path in [CONFIG['models_dir'], CONFIG['logs_dir'], CONFIG['plots_dir'], CONFIG['confusion_matrix_dir']]:
    os.makedirs(dir_path, exist_ok=True)

# Load preprocessed data
X_train = np.load(os.path.join(CONFIG['data_dir'], 'X_train.npy'))
X_val = np.load(os.path.join(CONFIG['data_dir'], 'X_val.npy'))
X_test = np.load(os.path.join(CONFIG['data_dir'], 'X_test.npy'))
y_train = np.load(os.path.join(CONFIG['data_dir'], 'y_train.npy'))
y_val = np.load(os.path.join(CONFIG['data_dir'], 'y_val.npy'))
y_test = np.load(os.path.join(CONFIG['data_dir'], 'y_test.npy'))

# Load class names
with open(os.path.join(CONFIG['data_dir'], 'label_encoding.json'), 'r') as f:
    label_encoding = json.load(f)
    class_names = [label_encoding[str(i)] for i in range(CONFIG['num_classes'])]

print(f"Data loaded successfully")
print(f"  Training set: {X_train.shape}")
print(f"  Validation set: {X_val.shape}")
print(f"  Test set: {X_test.shape}")
print(f"  Number of classes: {CONFIG['num_classes']}")

# Compute class weights for imbalanced dataset
class_weight = compute_class_weights_from_labels(y_train, class_names=class_names, verbose=True)

## 3. Build EfficientNetB0 Models with Custom Classification Heads

In [ ]:
# Build initial EfficientNetB0 model (frozen base, high learning rate for feature extraction)
print("=" * 80)
print("PHASE 1: Building EfficientNetB0 Model (Frozen Base, High Learning Rate)")
print("=" * 80)

efficientnetb0_model = build_efficientnet_b0_model(
    num_classes=CONFIG['num_classes'],
    input_shape=CONFIG['input_shape'],
    learning_rate=CONFIG['phase1_learning_rate'],
    optimizer=CONFIG['optimizer'],
    dense_units=CONFIG['dense_units'],
    dropout_rate=CONFIG['dropout_rate']
)

print("\nEfficientNetB0 Model Summary:")
efficientnetb0_model.summary()
print(f"\nTotal Parameters: {efficientnetb0_model.count_params():,}")
print(f"Trainable Parameters: {sum([tf.size(w).numpy() for w in efficientnetb0_model.trainable_weights]):,}")

## 4. Model Compilation Verification

In [ ]:
# Verify model compilation
print("\nModel Compilation Details:")
print(f"Optimizer: {efficientnetb0_model.optimizer.__class__.__name__}")
print(f"Loss Function: {efficientnetb0_model.loss}")
print(f"Metrics: {efficientnetb0_model.metrics}")
print(f"\nPhase 1 Learning Rate: {CONFIG['phase1_learning_rate']}")
print(f"Phase 2 Learning Rate: {CONFIG['phase2_learning_rate']}")

## 5. Data Preprocessing and Augmentation

In [ ]:
# Verify input data shapes and ranges
print("Input Data Verification:")
print(f"X_train shape: {X_train.shape}, dtype: {X_train.dtype}")
print(f"X_train range: [{X_train.min():.4f}, {X_train.max():.4f}]")
print(f"X_val shape: {X_val.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_val shape: {y_val.shape}")
print(f"y_test shape: {y_test.shape}")

# Create data augmentation pipeline for training
train_augmentation = keras.Sequential([
    layers.RandomRotation(0.2, input_shape=CONFIG['input_shape']),
    layers.RandomZoom(0.2),
    layers.RandomTranslation(0.1, 0.1),
    layers.RandomFlip("horizontal"),
    layers.GaussianNoise(0.02)
], name='augmentation')

print(f"\nData Augmentation Pipeline: {len(train_augmentation.layers)} layers")

# Prepare training data as tf.data.Dataset for efficient batching
train_dataset = tf.data.Dataset.from_tensor_slices((X_train, y_train)) \
    .shuffle(buffer_size=CONFIG['batch_size'] * 2) \
    .batch(CONFIG['batch_size']) \
    .prefetch(tf.data.AUTOTUNE)

val_dataset = tf.data.Dataset.from_tensor_slices((X_val, y_val)) \
    .batch(CONFIG['batch_size']) \
    .prefetch(tf.data.AUTOTUNE)

test_dataset = tf.data.Dataset.from_tensor_slices((X_test, y_test)) \
    .batch(CONFIG['batch_size']) \
    .prefetch(tf.data.AUTOTUNE)

print(f"\nDataset Pipeline Created:")
print(f"  Train batches: {len(train_dataset)}")
print(f"  Val batches: {len(val_dataset)}")
print(f"  Test batches: {len(test_dataset)}")

## 6. Define Training Callbacks

In [ ]:
# Create training callbacks for monitoring and early stopping
def create_callbacks(phase_name: str):
    """
    Create training callbacks for monitoring and early stopping.
    
    Args:
        phase_name: "phase1" or "phase2" to distinguish log files
    """
    callbacks = [
        # Early stopping to prevent overfitting
        keras.callbacks.EarlyStopping(
            monitor='val_loss',
            patience=8,
            restore_best_weights=True,
            verbose=1
        ),
        # Save best model based on validation accuracy
        keras.callbacks.ModelCheckpoint(
            filepath=f'{CONFIG["models_dir"]}/efficientnetb0_{phase_name}_best.h5',
            monitor='val_accuracy',
            save_best_only=True,
            verbose=0
        ),
        # Reduce learning rate on plateau
        keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.5,
            patience=3,
            min_lr=1e-7,
            verbose=1
        ),
        # Log to tensorboard
        keras.callbacks.TensorBoard(
            log_dir=f'{CONFIG["logs_dir"]}/efficientnetb0_{phase_name}',
            histogram_freq=1,
            write_graph=True
        )
    ]
    return callbacks

phase1_callbacks = create_callbacks('phase1')
print(f"Phase 1 Callbacks: {[cb.__class__.__name__ for cb in phase1_callbacks]}")

## 7. Phase 1 Training (Frozen Base, Feature Extraction)

In [ ]:
# Train EfficientNetB0 model - Phase 1 (frozen base)
print("=" * 80)
print("PHASE 1 TRAINING: EfficientNetB0 with Frozen Base")
print("=" * 80)
print(f"Epochs: {CONFIG['phase1_epochs']}")
print(f"Learning Rate: {CONFIG['phase1_learning_rate']}")
print(f"Batch Size: {CONFIG['batch_size']}")
print(f"Training Samples: {X_train.shape[0]}")
print("=" * 80)

efficientnetb0_history = efficientnetb0_model.fit(
    X_train, y_train,
    epochs=CONFIG['phase1_epochs'],
    batch_size=CONFIG['batch_size'],
    validation_data=(X_val, y_val),
    class_weight=class_weight,
    callbacks=phase1_callbacks,
    verbose=1
)

# Save training history
history_dict = {
    'loss': [float(x) for x in efficientnetb0_history.history['loss']],
    'accuracy': [float(x) for x in efficientnetb0_history.history['accuracy']],
    'val_loss': [float(x) for x in efficientnetb0_history.history['val_loss']],
    'val_accuracy': [float(x) for x in efficientnetb0_history.history['val_accuracy']]
}

with open(f'{CONFIG["logs_dir"]}/efficientnetb0_phase1_history.json', 'w') as f:
    json.dump(history_dict, f, indent=4)

print("\n" + "=" * 80)
print("PHASE 1 TRAINING COMPLETED")
print("=" * 80)
print(f"Final Train Loss: {efficientnetb0_history.history['loss'][-1]:.4f}")
print(f"Final Train Accuracy: {efficientnetb0_history.history['accuracy'][-1]:.4f}")
print(f"Final Val Loss: {efficientnetb0_history.history['val_loss'][-1]:.4f}")
print(f"Final Val Accuracy: {efficientnetb0_history.history['val_accuracy'][-1]:.4f}")

## 8. Evaluate Phase 1 Model Performance

In [ ]:
# Evaluate model on test set using model_utils
print("\n" + "=" * 80)
print("PHASE 1 EVALUATION ON TEST SET")
print("=" * 80)

metrics_phase1 = evaluate_model(
    efficientnetb0_model,
    X_test,
    y_test,
    class_names=class_names
)

print("\nPhase 1 Test Set Metrics:")
print(f"  Accuracy: {metrics_phase1['accuracy']:.4f}")
print(f"  Macro Precision: {metrics_phase1['macro_precision']:.4f}")
print(f"  Macro Recall: {metrics_phase1['macro_recall']:.4f}")
print(f"  Macro F1-Score: {metrics_phase1['macro_f1']:.4f}")
print(f"  Weighted F1-Score: {metrics_phase1['weighted_f1']:.4f}")

# Get confusion matrix for later visualization
cm_phase1 = metrics_phase1['confusion_matrix']
print(f"\nConfusion Matrix shape: {cm_phase1.shape}")

## 9. Generate Confusion Matrix and Class-wise Performance

In [ ]:
# Save confusion matrix
# Save confusion matrix as numpy file
cm_phase1_path = f'{CONFIG["confusion_matrix_dir"]}/efficientnetb0_phase1_cm.npy'
np.save(cm_phase1_path, cm_phase1)
print(f"Confusion matrix saved: {cm_phase1_path}")

# Calculate class-wise metrics from confusion matrix
def calculate_class_metrics(cm, class_names):
    """Calculate precision, recall, F1 for each class."""
    metrics = {}
    for i in range(len(class_names)):
        tp = cm[i, i]
        fp = cm[:, i].sum() - tp
        fn = cm[i, :].sum() - tp
        
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0
        f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
        
        metrics[class_names[i]] = {
            'precision': float(precision),
            'recall': float(recall),
            'f1_score': float(f1),
            'support': int(cm[i, :].sum())
        }
    return metrics

class_metrics_phase1 = calculate_class_metrics(cm_phase1, class_names)

# Save class metrics to JSON
with open(f'{CONFIG["logs_dir"]}/efficientnetb0_phase1_class_metrics.json', 'w') as f:
    json.dump(class_metrics_phase1, f, indent=4)

# Display top 5 best and worst classes by F1 score
f1_scores = {name: metrics['f1_score'] for name, metrics in class_metrics_phase1.items()}
sorted_f1 = sorted(f1_scores.items(), key=lambda x: x[1], reverse=True)

print("\nTop 5 Best Classes (by F1-Score):")
for class_name, f1 in sorted_f1[:5]:
    metrics = class_metrics_phase1[class_name]
    print(f"  {class_name:40s}: F1={f1:.4f}, Precision={metrics['precision']:.4f}, Recall={metrics['recall']:.4f}")

print("\nTop 5 Worst Classes (by F1-Score):")
for class_name, f1 in sorted_f1[-5:]:
    metrics = class_metrics_phase1[class_name]
    print(f"  {class_name:40s}: F1={f1:.4f}, Precision={metrics['precision']:.4f}, Recall={metrics['recall']:.4f}")

## 10. Plot Training History and Evaluation Metrics

In [ ]:
# Plot training history - Loss and Accuracy
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot Loss
axes[0].plot(efficientnetb0_history.history['loss'], label='Training Loss', linewidth=2)
axes[0].plot(efficientnetb0_history.history['val_loss'], label='Validation Loss', linewidth=2)
axes[0].set_title('EfficientNetB0 Phase 1: Loss', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Plot Accuracy
axes[1].plot(efficientnetb0_history.history['accuracy'], label='Training Accuracy', linewidth=2)
axes[1].plot(efficientnetb0_history.history['val_accuracy'], label='Validation Accuracy', linewidth=2)
axes[1].set_title('EfficientNetB0 Phase 1: Accuracy', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{CONFIG["plots_dir"]}/efficientnetb0_phase1_history.png', dpi=300, bbox_inches='tight')
plt.show()

print("Training history plot saved")

# Plot Confusion Matrix for Phase 1
plt.figure(figsize=(16, 14))
sns.heatmap(cm_phase1, annot=False, fmt='d', cmap='Blues', cbar=True,
            xticklabels=class_names, yticklabels=class_names, cbar_kws={'label': 'Count'})
plt.title('EfficientNetB0 Phase 1: Confusion Matrix', fontsize=14, fontweight='bold', pad=20)
plt.xlabel('Predicted Label', fontsize=12)
plt.ylabel('True Label', fontsize=12)
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(rotation=0, fontsize=8)
plt.tight_layout()
plt.savefig(f'{CONFIG["confusion_matrix_dir"]}/efficientnetb0_phase1_cm.png', dpi=300, bbox_inches='tight')
plt.show()

print("Confusion matrix plot saved")

# Plot class-wise F1 scores
f1_scores_sorted = dict(sorted(f1_scores.items(), key=lambda x: x[1]))
plt.figure(figsize=(12, 8))
colors = ['red' if f1 < 0.2 else 'orange' if f1 < 0.5 else 'green' for f1 in f1_scores_sorted.values()]
plt.barh(range(len(f1_scores_sorted)), list(f1_scores_sorted.values()), color=colors)
plt.yticks(range(len(f1_scores_sorted)), list(f1_scores_sorted.keys()), fontsize=8)
plt.xlabel('F1-Score', fontsize=12)
plt.title('EfficientNetB0 Phase 1: Class-wise F1-Scores', fontsize=14, fontweight='bold')
plt.xlim(0, 1)
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig(f'{CONFIG["plots_dir"]}/efficientnetb0_phase1_f1_scores.png', dpi=300, bbox_inches='tight')
plt.show()

print("Class-wise F1 scores plot saved")

## 11. Fine-Tuning Strategy: Unfreeze and Train with Lower Learning Rate

In [ ]:
# Fine-tuning strategy: Unfreeze last N layers of EfficientNetB0 base model
print("\n" + "=" * 80)
print("PHASE 2: FINE-TUNING STRATEGY")
print("=" * 80)

# Count base model layers
base_model_layers = [layer for layer in efficientnetb0_model.layers if 'efficientnet' in layer.name]
print(f"\nBase model layer count: {len(base_model_layers)}")
print(f"Unfreezing last {CONFIG['num_unfreeze_layers']} layers for fine-tuning")
print(f"Phase 2 Learning Rate: {CONFIG['phase2_learning_rate']}")

# Unfreeze last N layers using the utility function
unfreeze_model_layers(
    efficientnetb0_model,
    num_layers=CONFIG['num_unfreeze_layers'],
    learning_rate=CONFIG['phase2_learning_rate'],
    optimizer=CONFIG['optimizer']
)

# Verify which layers are trainable
trainable_layers = [layer.name for layer in efficientnetb0_model.layers if layer.trainable]
print(f"\nTrainable layers: {len(trainable_layers)}")
print(f"Total parameters: {efficientnetb0_model.count_params():,}")

# Check trainable parameter count
total_params = sum([tf.size(w).numpy() for w in efficientnetb0_model.trainable_weights])
print(f"Trainable parameters: {total_params:,}")

## 12. Phase 2 Training (Fine-Tuning with Unfrozen Layers)

In [ ]:
# Train Phase 2 (fine-tuning with unfrozen layers)
print("\n" + "=" * 80)
print("PHASE 2 TRAINING: EfficientNetB0 with Fine-Tuning")
print("=" * 80)
print(f"Epochs: {CONFIG['phase2_epochs']}")
print(f"Learning Rate: {CONFIG['phase2_learning_rate']}")
print(f"Batch Size: {CONFIG['batch_size']}")
print("=" * 80)

phase2_callbacks = create_callbacks('phase2')

efficientnetb0_history_phase2 = efficientnetb0_model.fit(
    X_train, y_train,
    epochs=CONFIG['phase2_epochs'],
    batch_size=CONFIG['batch_size'],
    validation_data=(X_val, y_val),
    class_weight=class_weight,
    callbacks=phase2_callbacks,
    verbose=1
)

# Save phase 2 training history
history_dict_phase2 = {
    'loss': [float(x) for x in efficientnetb0_history_phase2.history['loss']],
    'accuracy': [float(x) for x in efficientnetb0_history_phase2.history['accuracy']],
    'val_loss': [float(x) for x in efficientnetb0_history_phase2.history['val_loss']],
    'val_accuracy': [float(x) for x in efficientnetb0_history_phase2.history['val_accuracy']]
}

with open(f'{CONFIG["logs_dir"]}/efficientnetb0_phase2_history.json', 'w') as f:
    json.dump(history_dict_phase2, f, indent=4)

print("\n" + "=" * 80)
print("PHASE 2 TRAINING COMPLETED")
print("=" * 80)
print(f"Final Train Loss: {efficientnetb0_history_phase2.history['loss'][-1]:.4f}")
print(f"Final Train Accuracy: {efficientnetb0_history_phase2.history['accuracy'][-1]:.4f}")
print(f"Final Val Loss: {efficientnetb0_history_phase2.history['val_loss'][-1]:.4f}")
print(f"Final Val Accuracy: {efficientnetb0_history_phase2.history['val_accuracy'][-1]:.4f}")

## 13. Evaluate Phase 2 Model and Compare with Phase 1

In [ ]:
# Evaluate Phase 2 model on test set
print("\n" + "=" * 80)
print("PHASE 2 EVALUATION ON TEST SET")
print("=" * 80)

metrics_phase2 = evaluate_model(
    efficientnetb0_model,
    X_test,
    y_test,
    class_names=class_names
)

print("\nPhase 2 Test Set Metrics:")
print(f"  Accuracy: {metrics_phase2['accuracy']:.4f}")
print(f"  Macro Precision: {metrics_phase2['macro_precision']:.4f}")
print(f"  Macro Recall: {metrics_phase2['macro_recall']:.4f}")
print(f"  Macro F1-Score: {metrics_phase2['macro_f1']:.4f}")
print(f"  Weighted F1-Score: {metrics_phase2['weighted_f1']:.4f}")

cm_phase2 = metrics_phase2['confusion_matrix']

# Compare Phase 1 vs Phase 2
print("\n" + "=" * 80)
print("PHASE 1 vs PHASE 2 COMPARISON")
print("=" * 80)
print(f"{'Metric':<25} {'Phase 1':<15} {'Phase 2':<15} {'Improvement':<15}")
print("-" * 70)

metrics_to_compare = [
    ('Accuracy', 'accuracy'),
    ('Macro Precision', 'macro_precision'),
    ('Macro Recall', 'macro_recall'),
    ('Macro F1-Score', 'macro_f1'),
    ('Weighted F1-Score', 'weighted_f1')
]

for metric_name, metric_key in metrics_to_compare:
    phase1_val = metrics_phase1[metric_key]
    phase2_val = metrics_phase2[metric_key]
    improvement = phase2_val - phase1_val
    improvement_pct = (improvement / phase1_val * 100) if phase1_val > 0 else 0
    
    print(f"{metric_name:<25} {phase1_val:<15.4f} {phase2_val:<15.4f} {improvement:+.4f} ({improvement_pct:+.1f}%)")

# Plot comparison of training histories
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot Phase 1 Loss
axes[0, 0].plot(efficientnetb0_history.history['loss'], label='Training', linewidth=2)
axes[0, 0].plot(efficientnetb0_history.history['val_loss'], label='Validation', linewidth=2)
axes[0, 0].set_title('Phase 1: Loss', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Loss')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Plot Phase 1 Accuracy
axes[0, 1].plot(efficientnetb0_history.history['accuracy'], label='Training', linewidth=2)
axes[0, 1].plot(efficientnetb0_history.history['val_accuracy'], label='Validation', linewidth=2)
axes[0, 1].set_title('Phase 1: Accuracy', fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('Accuracy')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Plot Phase 2 Loss
axes[1, 0].plot(efficientnetb0_history_phase2.history['loss'], label='Training', linewidth=2)
axes[1, 0].plot(efficientnetb0_history_phase2.history['val_loss'], label='Validation', linewidth=2)
axes[1, 0].set_title('Phase 2: Loss', fontsize=12, fontweight='bold')
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('Loss')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Plot Phase 2 Accuracy
axes[1, 1].plot(efficientnetb0_history_phase2.history['accuracy'], label='Training', linewidth=2)
axes[1, 1].plot(efficientnetb0_history_phase2.history['val_accuracy'], label='Validation', linewidth=2)
axes[1, 1].set_title('Phase 2: Accuracy', fontsize=12, fontweight='bold')
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('Accuracy')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{CONFIG["plots_dir"]}/efficientnetb0_phase1_vs_phase2_history.png', dpi=300, bbox_inches='tight')
plt.show()

print("\nPhase 1 vs Phase 2 comparison plot saved")

# Save confusion matrix for Phase 2
np.save(f'{CONFIG["confusion_matrix_dir"]}/efficientnetb0_phase2_cm.npy', cm_phase2)
print(f"Phase 2 confusion matrix saved")

## 14. Save and Load Models

In [ ]:
# Save final EfficientNetB0 model
print("\n" + "=" * 80)
print("SAVING MODELS")
print("=" * 80)

# Save as .h5 format (includes architecture and weights)
final_model_path = f'{CONFIG["models_dir"]}/efficientnetb0_final.h5'
efficientnetb0_model.save(final_model_path)
print(f"✓ Final model saved: {final_model_path}")

# Save as SavedModel format (recommended for production)
savedmodel_path = f'{CONFIG["models_dir"]}/efficientnetb0_final_savedmodel'
efficientnetb0_model.save(savedmodel_path, save_format='tf')
print(f"✓ SavedModel saved: {savedmodel_path}")

# Save model configuration and metadata
metadata = {
    'architecture': 'EfficientNetB0',
    'num_classes': CONFIG['num_classes'],
    'input_shape': CONFIG['input_shape'],
    'total_parameters': int(efficientnetb0_model.count_params()),
    'trainable_parameters': int(sum([tf.size(w).numpy() for w in efficientnetb0_model.trainable_weights])),
    'phase1_epochs': CONFIG['phase1_epochs'],
    'phase2_epochs': CONFIG['phase2_epochs'],
    'batch_size': CONFIG['batch_size'],
    'final_accuracy': float(metrics_phase2['accuracy']),
    'final_loss': float(efficientnetb0_history_phase2.history['val_loss'][-1]),
    'class_names': class_names
}

with open(f'{CONFIG["models_dir"]}/efficientnetb0_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=4)
print(f"✓ Model metadata saved")

# Load and verify model
print("\n" + "=" * 80)
print("VERIFYING SAVED MODEL")
print("=" * 80)

loaded_model = keras.models.load_model(final_model_path)
print(f"✓ Model loaded successfully")
print(f"  Loaded model input shape: {loaded_model.input_shape}")
print(f"  Loaded model output shape: {loaded_model.output_shape}")

# Test prediction with loaded model
sample_predictions = loaded_model.predict(X_test[:5], verbose=0)
print(f"✓ Model predictions verified")
print(f"  Sample prediction shape: {sample_predictions.shape}")
print(f"  Sample prediction values (first image, top 5 classes):")
top_5_indices = np.argsort(sample_predictions[0])[::-1][:5]
for idx in top_5_indices:
    print(f"    {class_names[idx]}: {sample_predictions[0][idx]:.4f}")

print("\n" + "=" * 80)
print("EFFICIENTNETB0 TRANSFER LEARNING TRAINING COMPLETE")
print("=" * 80)
print(f"\nFinal Metrics:")
print(f"  Test Accuracy: {metrics_phase2['accuracy']:.4f}")
print(f"  Test F1-Score (Weighted): {metrics_phase2['weighted_f1']:.4f}")
print(f"  Total training time: ~{CONFIG['phase1_epochs'] + CONFIG['phase2_epochs']} epochs")